In [65]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import faiss
import spacy
import re
import contractions
from textblob import TextBlob

## 1. Load the document(.txt)

In [66]:
data = open('data.txt').read()

## 2. Text Normalization

### Converting all the characters into lower case

In [67]:
data = data.lower()

### Removing extra spaces

In [68]:
data = re.sub(r'\s{2,}','',data)

### Removing numbers like 1.,2.

In [69]:
data = re.sub(r'\d+\.','',data)
data

' machine learning uses support vector machines to support supply chain optimization while improving loss functions in modern systems.\n in sentiment analysis, deep learning benefits from k nearest neighbors because it strengthens activation functions and improves decision quality.\n neural networks often applies naive bayes to document summarization, where data augmentation helps the model generalize more effectively.\n a practical study of natural language processing shows that principal component analysis can improve video understanding by refining class imbalance during training.\n researchers use computer vision together with convolutional neural networks to solve personalized tutoring and better understand overfitting control.\n when reinforcement learning is combined with recurrent neural networks, practitioners can address smart agriculture with more reliable model interpretability.\n recommendation systems and transformers are frequently paired in traffic prediction, with onli

### Contractions

In [70]:
data = contractions.fix(data)

### Removing punctuations & spl characters

In [71]:
data = re.sub(r'[^0-9a-zA-Z\s]','',data)

### TextBlob

In [72]:
# values= TextBlob(data).correct()
# values

### Lemmatization

In [73]:
import spacy

nlp = spacy.load('en_core_web_sm')
tokens = nlp(data)
updated_tokens = [token.lemma_ for token in tokens if not token.is_stop]
data = ' '.join(updated_tokens).strip()
data

'machine learning use support vector machine support supply chain optimization improve loss function modern system \n  sentiment analysis deep learning benefit k near neighbor strengthen activation function improve decision quality \n  neural network apply naive baye document summarization datum augmentation help model generalize effectively \n  practical study natural language processing show principal component analysis improve video understanding refine class imbalance training \n  researcher use computer vision convolutional neural network solve personalized tutoring well understand overfitte control \n  reinforcement learning combine recurrent neural network practitioner address smart agriculture reliable model interpretability \n  recommendation system transformer frequently pair traffic prediction online learning guide learning process \n  cybersecurity monitoring time series forecasting leverage long short term memory feature engineering build adaptable accurate model \n  anoma

### Chunking(Converting doc -> chunks)

In [74]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap = 40
)

chunks = list(set(splitter.split_text(data)))

In [75]:
print(len(chunks))

74


### Chunk Embeddings (Converts chunks to vectors)

In [76]:
embedding_model = SentenceTransformer(
    model_name_or_path='sentence-transformers/all-miniLM-L6-V2'
)
chunk_embeddings=embedding_model.encode(chunks).astype('float32')

In [77]:
chunk_embeddings.shape

(74, 384)

In [78]:
dimension = chunk_embeddings.shape[1]
dimension

384

In [79]:
faiss.normalize_L2(chunk_embeddings)

In [80]:
index_faiss_db = faiss.IndexFlatIP(dimension)
index_faiss_db.add(chunk_embeddings)

In [ ]:
def r_search(query,k=3):
    query_embeddings = embedding_model.encode(query).astype('float32')
    query_embeddings = query_embeddings.reshape(1,-1)
    faiss.normalize_L2(query_embeddings)
    print(query_embeddings.shape)
    distance,index = index_faiss_db.search(query_embeddings,k=k)
    R_chunks = [chunks[i] for i in index[0]]
    R_str = ' '.join(R_chunks)
    return R_str
def g_text(r_search):
        import os
        import requests
    
        API_URL = "https://router.huggingface.co/v1/chat/completions"
    
        headers = {
            "Authorization": f"Bearer {os.environ['HF_TOKEN']}",
        }
        def query(payload):
            response = requests.post(API_URL, headers=headers, json=payload)
            return response.json()
        prompt = f'''
                    You're an helpful assistant
                    Assigned Task for you : Structure my output => {r_search}
                    Note : 
                    1) Don't add extra contents just structure mentioned output.
                    2) If there is mistake in output correct or else keep the original output
                    with structured result.
            '''
        response = query({
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            "model": "deepseek-ai/DeepSeek-R1:novita"
        })
    
        return response
user_prompt = 'Explain Machine Learning ?'
user_prompt = re.sub(r'[^0-9a-zA-Z\s]','',user_prompt)

r_response = r_search(user_prompt)
g_response = g_text(r_response)
print(g_response)

(1, 384)
